**Import libraries**

In [4]:
import os
os.chdir("/exp/sbnd/app/users/svidales/AI_nuvT")

In [5]:
from imports import *

2025-04-24 23:34:49.776858: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:10575] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-04-24 23:34:49.776950: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:479] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-04-24 23:34:49.778132: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1442] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-04-24 23:34:49.783054: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


## 1. Preprocessing

The data corresponds to a MC simulation of the SBND experiment used in the paper "Scintillation Light in SBND: Simulation, Reconstruction, and Expected Performance of the Photon Detection System" in https://arxiv.org/abs/2406.07514. It is a simulation of BNB + Cosmics and their subsequent interaction in SBND, as well as the simulation of the detector's response to the particles resulting from the interaction of the neutrinos.

### Load variables

**archivos en espacio personal - a partir de ahora correrlos en data**

In [3]:
file_path = '/exp/sbnd/data/users/svidales/opana_tree_combined_v2304_complete.root'
file = uproot.open(file_path)
optree = file['opanatree']['OpAnaTree'] # Tree con número de fotoelectrones
print("Keys in optree:", optree.keys())

Keys in optree: ['eventID', 'runID', 'subrunID', 'nuvX', 'nuvY', 'nuvZ', 'nuvT', 'nuvE', 'stepX', 'stepY', 'stepZ', 'stepT', 'dE', 'energydep', 'energydepX', 'energydepY', 'energydepZ', 'E', 'StartPx', 'StartPy', 'StartPz', 'EndPx', 'EndPy', 'EndPz', 'process', 'trackID', 'motherID', 'PDGcode', 'InTimeCosmics', 'InTimeCosmicsTime', 'dEtpc', 'dEpromx', 'dEpromy', 'dEpromz', 'dEspreadx', 'dEspready', 'dEspreadz', 'dElowedges', 'dEmaxedges', 'nopflash', 'flash_id', 'flash_time', 'flash_total_pe', 'flash_pe_v', 'flash_tpc', 'flash_y', 'flash_yerr', 'flash_z', 'flash_zerr', 'flash_x', 'flash_xerr', 'flash_ophit_time', 'flash_ophit_risetime', 'flash_ophit_starttime', 'flash_ophit_amp', 'flash_ophit_area', 'flash_ophit_width', 'flash_ophit_pe', 'flash_ophit_ch']


In [4]:
# Input variables
# f_ophit_PE: number of photoelectrons (PEs) per OpHit
# f_ophit_ch: number of channel that collect the OpHit
# f_ophit_t: OpHit time

# Labels
# nuvT: neutrino interaction time
# dEprom{x,y,z}: energy barycenter coordinates {x,y,z}

# Auxiliary 
# dEtpc: allows filtering by energy deposited in the TPC

# These are awkward arrays (i.e., irregular structures) with a 3-level hierarchy (events -> flashes -> hits)

f_ophit_PE, f_ophit_ch, f_ophit_t, nuvT, dEpromx, dEpromy, dEpromz, dEtpc, nuvZ = (
    optree[key].array() for key in 
    ['flash_ophit_pe', 'flash_ophit_ch', 'flash_ophit_time', 'nuvT', 'dEpromx', 'dEpromy', 'dEpromz', 'dEtpc', "nuvZ"]
)

In [5]:
print(len(f_ophit_PE))

243599


**Eliminate events with more than one neutrino & events with no flashes**

In [3]:
# Filter events where nuvT has exactly one element
mask = ak.num(nuvT) == 1  

# Apply the mask to all variables
f_ophit_PE_1, f_ophit_ch_1, f_ophit_t_1 = f_ophit_PE[mask], f_ophit_ch[mask], f_ophit_t[mask]
nuvT_1, dEpromx_1, dEpromy_1, dEpromz_1, dEtpc_1, nuvZ_1 = nuvT[mask], dEpromx[mask], dEpromy[mask], dEpromz[mask], dEtpc[mask], nuvZ[mask]

NameError: name 'ak' is not defined

In [7]:
# Filter events with at least one flash
mask = ak.num(f_ophit_PE_1, axis=1) > 0  

# Apply the mask to all variables
f_ophit_PE_2, f_ophit_ch_2, f_ophit_t_2 = f_ophit_PE_1[mask], f_ophit_ch_1[mask], f_ophit_t_1[mask]
nuvT_2, dEpromx_2, dEpromy_2, dEpromz_2, dEtpc_2, nuvZ_2 = nuvT_1[mask], dEpromx_1[mask], dEpromy_1[mask], dEpromz_1[mask], dEtpc_1[mask], nuvZ_1[mask]


### Corrección PTM delay

**Correction PMT delay 135 ns due to the difference between the photon arrival times (at the photocathode)
and the digitised signal (at the anode)**

**es posible que luego añada las coordenadas en la siguiente celda y lo guarde**

In [2]:
PDSMap = file['opanatree']['PDSMapTree']
ID = PDSMap['OpDetID'].array()
Type = PDSMap['OpDetType'].array()
channel_dict = {id_val: (int(type_val)) for id_val, type_val in zip(ID[0],Type[0])}
print(channel_dict)

NameError: name 'file' is not defined

In [1]:
# Create the list of channels to correct
channels_to_correct = [ch for ch, value in channel_dict.items() if value in {0, 1}]
print(channels_to_correct)

# Create a mask for the channels to correct
mask = ak.Array([
    [[ch in channels_to_correct for ch in ophit] for ophit in flash] for flash in f_ophit_ch_2
])

# Apply the mask to f_ophit_t variable
f_ophit_t_adj = ak.where(mask, f_ophit_t_2 - 0.135, f_ophit_t_2)

NameError: name 'channel_dict' is not defined

### Selección de TPC para la variable dEprom{x,y,z} y (Opcional) para los flashes de las variables flash_ophit

In [ ]:
import awkward as ak
import numpy as np

# --- 1. Clasificación de canales ---
pmt_channels = [ch for ch, val in channel_dict.items() if val in {0, 1}]
xas_channels = [ch for ch, val in channel_dict.items() if val in {2, 3}]

def split_even_odd(channels):
    return set(filter(lambda x: x % 2 == 0, channels)), set(filter(lambda x: x % 2 != 0, channels))

pmt_even, pmt_odd = split_even_odd(pmt_channels)
xas_even, xas_odd = split_even_odd(xas_channels)

def categorize_first_ch(vector):
    if isinstance(vector, (list, ak.Array)) and len(vector) > 0:
        ch = vector[0]
        if ch in pmt_even: return 0
        if ch in pmt_odd:  return 1
        if ch in xas_even: return 2
        if ch in xas_odd:  return 3
    return -1  # No clasificado o vacío

categorized_flashes = ak.Array([
    [categorize_first_ch(flash) for flash in event]
    for event in ak.to_list(f_ophit_ch_2)
])

# --- 2. Sumar los PEs de cada flash ---
sum_pe = ak.sum(f_ophit_PE_2, axis=2)  # [evento][flash]

# --- 3. Crear máscaras por categoría ---
mask_even = (categorized_flashes == 0) | (categorized_flashes == 2)
mask_odd  = (categorized_flashes == 1) | (categorized_flashes == 3)

# --- 4. Sumar PE por grupo ---
sum_even = ak.sum(ak.where(mask_even, sum_pe, 0), axis=1)
sum_odd  = ak.sum(ak.where(mask_odd, sum_pe, 0), axis=1)

# --- 5. Selección del grupo con mayor PE si hay más de 2 flashes ---
n_flashes = ak.num(categorized_flashes)
decision = sum_even >= sum_odd 

# Generar máscara de selección
selected_mask = ak.Array([
    np.ones(n, dtype=bool) if n <= 2  # Keep all flashes for ≤ 2
    else (mask_even[i] if decision[i] else mask_odd[i])
    for i, n in enumerate(ak.to_list(n_flashes))
])

# --- 6. Función para filtrar un array usando la máscara ---
def filter_by_mask(array, mask):
    return ak.Array([
        [item for item, flag in zip(event, event_mask) if flag]
        for event, event_mask in zip(ak.to_list(array), ak.to_list(mask))
    ])

# --- 7. Aplicar máscaras de selección ---
f_ophit_t_adj_sel       = filter_by_mask(f_ophit_t_adj, selected_mask)
f_ophit_PE_2_sel        = filter_by_mask(f_ophit_PE_2, selected_mask)
f_ophit_ch_2_sel        = filter_by_mask(f_ophit_ch_2, selected_mask)
categorized_flashes_sel = filter_by_mask(categorized_flashes, selected_mask)

# --- 8. Selección TPC ganadora ---
selector = ak.Array([[d, not d] for d in decision])

def select_tpc_value(array_2d, selector):
    return ak.sum(ak.where(selector, array_2d, 0), axis=1)

dEpromx_sel = select_tpc_value(dEpromx_2, selector)
dEpromy_sel = select_tpc_value(dEpromy_2, selector)
dEpromz_sel = select_tpc_value(dEpromz_2, selector)
dEtpc_sel   = select_tpc_value(dEtpc_2, selector)

In [ ]:
print("flashes iniciales:", categorized_flashes)
print("flashes que selecciona:", categorized_flashes_sel)
print("before TPC selection:", dEpromx_2)
print("after TPC selection:", dEpromx_sel)

flashes iniciales: [[0, 1, 2, 3], [0, 2], [0, 1, 2], [0, ...], ..., [1, 3], [0, 1, 3], [0, 1, 3]]
flashes que selecciona: [[0, 2], [0, 2], [0, 2], [1, 3], [1, ...], ..., [1, 3], [1, 3], [1, 3], [1, 3]]
before TPC selection: [[-189, 158], [-188, -999], [-133, -999], ..., [-26.4, 81.7], [-999, 115]]
after TPC selection: [-189, -188, -133, 146, 167, -96.1, -169, ..., 53.4, -113, 192, 28.9, 81.7, 115]


In [ ]:
# Create a boolean mask where dEpromx_f_unique is not -999 & select events with deposition >200 MeV (dEtpc_f > 200)

mask = (dEpromx_sel != -999) & (dEpromy_sel != -999) & (dEpromz_sel != -999) & (dEtpc_sel > 200)
mask_1d = ak.to_numpy(mask)

# Apply the mask to both the image and dEpromx_f_unique to keep only the valid entries

nuvT_3 = nuvT_2[mask_1d]
f_ophit_PE_3 = f_ophit_PE_2_sel[mask_1d]
f_ophit_ch_3 = f_ophit_ch_2_sel[mask_1d]
f_ophit_t_3 = f_ophit_t_adj_sel[mask_1d]
dEpromx_3 = dEpromx_sel[mask_1d]
dEpromy_3 = dEpromy_sel[mask_1d]
dEpromz_3 = dEpromz_sel[mask_1d]
dEtpc_3 = dEtpc_sel[mask_1d]
nuvZ_3 = nuvZ_2[mask_1d]

In [ ]:
# Diccionario con los arrays
data = {
    "nuvT": nuvT_3,
    "f_ophit_PE": f_ophit_PE_3,
    "f_ophit_ch": f_ophit_ch_3,
    "f_ophit_t": f_ophit_t_3,
    "dEpromx": dEpromx_3,
    "dEpromy": dEpromy_3,
    "dEpromz": dEpromz_3,
    "nuvZ": nuvZ_3,
}

# Convertir a Arrow Table y guardar como Parquet
table = ak.to_arrow_table(data)
pq.write_table(table, "/exp/sbnd/app/users/svidales/AI_nuvT/v2503_tpcselection_preproc_springval_allevents.parquet")